# GridSight 01 · The NEM market story

Exploratory analysis of the National Electricity Market: the duck curve, negative prices, price spikes
and how accurate AEMO's pre-dispatch forecasts are. Every number comes from `gridsight.analysis.market`,
so the same code feeds this notebook, `gridsight eda` and the dashboard.

If no real data has been loaded yet (`gridsight backfill ...`), the notebook loads synthetic data so it still runs.

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd

from gridsight import storage
from gridsight.analysis import market
from gridsight.config import get_settings

pd.set_option("display.float_format", "{:,.2f}".format)
THRESHOLD = get_settings().spike_threshold

price = storage.read("dispatch_price")
if price.empty:
    from gridsight.pipeline import load_synthetic
    from gridsight.synthetic import SyntheticConfig
    print("No silver data found - loading synthetic data for a demo run.")
    load_synthetic(SyntheticConfig(days=90))
    price = storage.read("dispatch_price")
demand = storage.read("region_demand")
pre = storage.read("predispatch")
print(f"{len(price):,} price rows, {price['region'].nunique()} regions, "
      f"{price['interval_utc'].min():%Y-%m-%d} to {price['interval_utc'].max():%Y-%m-%d}")

## 1. Market summary

In [ ]:
market.market_summary(price, demand, THRESHOLD)

## 2. The duck curve

Rooftop solar pushes midday operational demand down while the evening peak stays. The depth of the "duck"
(evening peak minus midday trough) grows as rooftop PV capacity grows.

In [ ]:
curve = market.duck_curve(demand, by="season")
fig, axes = plt.subplots(1, curve["region"].nunique(), figsize=(16, 3.5), sharex=True)
for ax, (region, g) in zip(axes if hasattr(axes, "__len__") else [axes], curve.groupby("region")):
    for season, s in g.groupby("season"):
        ax.plot(s["hour"], s["total_demand_mw"], label=season)
    ax.set_title(region); ax.set_xlabel("hour (market time)")
axes[0].set_ylabel("MW") if hasattr(axes, "__len__") else None
plt.legend(); plt.tight_layout(); plt.show()
market.duck_depth(demand, by="year")

## 3. Negative prices

In [ ]:
market.negative_prices(price, by="month").pivot(index="month", columns="region", values="share")

## 4. Price spikes: rare, but where the money is

In [ ]:
display(market.spike_stats(price, THRESHOLD))
by_hour = market.spikes_by_hour(price, THRESHOLD).pivot(index="hour", columns="region", values="spike_share")
by_hour.plot(figsize=(10, 3.5), title=f"Share of intervals above ${THRESHOLD:,.0f}/MWh by hour"); plt.show()

In [ ]:
pdc = market.price_duration_curve(price)
fig, ax = plt.subplots(figsize=(10, 3.5))
for region, g in pdc.groupby("region"):
    ax.plot(g["share_of_time"], g["rrp"], label=region)
ax.set_yscale("symlog"); ax.set_xlabel("share of time"); ax.set_ylabel("$/MWh (symlog)")
ax.set_title("Price duration curve"); ax.legend(); plt.show()

## 5. How good is AEMO pre-dispatch?

This is the benchmark GridSight has to beat. Error grows with horizon, and spikes are often missed.

In [ ]:
if pre.empty:
    print("No pre-dispatch data loaded.")
else:
    acc = market.predispatch_accuracy(pre, price, demand, THRESHOLD)
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    for region, g in acc.groupby("region"):
        axes[0].plot(g["horizon_steps"] / 2, g["price_mae"], label=region)
        axes[1].plot(g["horizon_steps"] / 2, g["demand_mape"] * 100, label=region)
    axes[0].set(title="Price MAE", xlabel="horizon (h)", ylabel="$/MWh")
    axes[1].set(title="Demand MAPE", xlabel="horizon (h)", ylabel="%")
    axes[1].legend(); plt.tight_layout(); plt.show()
    display(acc.groupby("region")[["price_mae", "demand_mape", "spike_recall"]].mean())

## 6. Findings

Write 3-5 findings here once real data is loaded, each with its number, for example:

- Midday demand in SA1 fell by X MW between 2019 and 2026 while the evening peak held.
- Negative prices occurred in X% of intervals in SA1 in spring 2025.
- Spikes above $300/MWh were X% of intervals but Y% of energy cost in SA1.
- AEMO pre-dispatch price MAE doubles between the 1-hour and 8-hour horizons.